# Data collection

Imports

In [1]:
import pandas as pd
from pathlib import Path
import yfinance as yf

### Step 1: Convert historical membership data to historical tickers dataframe & store in one file.

In [2]:
data_folder_path = Path("data")
membership_snapshots_path = data_folder_path / "membership_snapshots"

# Create the folder and its parents if they don't exist
membership_snapshots_path.mkdir(parents=True, exist_ok=True)

In [3]:
series_list = []

# The files have no extension
for file_path in sorted(membership_snapshots_path.glob("*")):
    s = pd.read_csv(
        file_path,
        sep=r"\t+",
        engine="python",
        header=None,
        usecols=[0]
    )[0]

    s = s.str.strip().str.replace('/', '-', regex=False) # fix ticker formatting to match yfinance format

    s.name = file_path.stem[3:] # remove the "spy" prefix, leaving the name as YYYYMM
    series_list.append(s)

membership_df = pd.concat(series_list, axis=1)

Save to CSV

In [4]:
membership_df.to_csv(data_folder_path / "membership_df.csv", index=False)

### Step 2: Store historical OHLCV price data

Get all unique tickers from January 2016 to January 2026

In [5]:
unique_tickers = pd.Series(membership_df.values.ravel()).dropna().unique().tolist()

Pull data for these tickers from January 2014 to January 2026 (2 extra years as a loopback period)

In [6]:
start_date = "2014-01-01"
end_date = "2026-01-03" # first trading day in 2026; yfinance does not include the end date

In [7]:
historical_ohlcv_df = yf.download(unique_tickers, start=start_date, end=end_date, group_by='ticker', auto_adjust=True) # auto adjust=True to account for splits and dividends

$ATVI: possibly delisted; no timezone found
[                       0%                       ]HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: LVLT"}}}
$LVLT: possibly delisted; no timezone found
[                       0%                       ]  2 of 705 completed$DWDP: possibly delisted; no timezone found
[                       1%                       ]  4 of 705 completed$NFX: Data doesn't exist for startDate = 1388552400, endDate = 1767416400
[**                     5%                       ]  32 of 705 completed$BHI: possibly delisted; no timezone found
[**                     5%                       ]  33 of 705 completed$SWN: possibly delisted; no timezone found
[**                     5%                       ]  35 of 705 completed$SPLS: Data doesn't exist for startDate = 1388552400, endDate = 1767416400
[***                    6%                       ]  45 of 705 completed$AGN: possibly delisted; no time

Save to parquet file (good for handling MultiIndex colums)

In [8]:
historical_ohlcv_df.to_parquet(
    data_folder_path / "historical_ohlcv_df.parquet"
)

In [9]:
historical_ohlcv_df.index[-1]

Timestamp('2026-01-02 00:00:00')